# Person 2 — Planner and Qwen Integration Handoff

**AAI-520 | Roopleen Kaur | Integration baseline: `de11239`**

This notebook provides a reproducible execution example for the team's existing financial research implementation. `ResearchPlanner` coordinates market, earnings, and news evidence retrieval (including FinBERT sentiment), then produces a Qwen-backed draft.

**Integration contract:** The notebook supplies `ticker`, `research_question`, and optional `lessons` to `ResearchPlanner.execute()`. The returned object contains `plan`, `retrieved_lessons`, `tool_trace`, `evidence`, and `draft_analysis`. These fields can be consumed by the final team notebook and the separate evaluation/revision workflow.

**Runtime:** Python environment with project dependencies; CUDA-enabled GPU recommended for Qwen inference. This handoff does not implement Ashley's evaluator or persistent-memory layer.

## 1. Reproducible integration baseline

The execution environment uses the team's integration branch at commit `de11239`, which includes the planner's optional `lessons` parameter. The isolated checkout prevents modifications to other working branches. In the final combined notebook, the existing integration checkout can replace this standalone setup.

In [1]:
from pathlib import Path
import subprocess, sys

REPO = Path('/content/NLP_AAI520-person2-handoff')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/tim17hays/NLP_AAI520.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'codex/tim-integration-handoff'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', 'de11239'], cwd=REPO, check=True)
print('Integration baseline:', subprocess.check_output(['git','rev-parse','--short','HEAD'], cwd=REPO, text=True).strip())

Integration baseline: de11239


## 2. Runtime dependencies

Dependencies are installed from the repository requirements file when present. Qwen and FinBERT weights are obtained by the existing project implementation during model initialization. GPU availability is reported for reproducibility.

In [2]:
import subprocess, sys
requirements = REPO / 'requirements.txt'
if requirements.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
else:
    print('No requirements.txt found; install project dependencies according to repository README.')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 3. Planner import and API compatibility

The integration checkout is prioritized on the Python import path. Previously loaded `src` modules are cleared to avoid mixing module versions from separate Colab checkouts. The signature check verifies support for the optional `lessons` argument.

In [3]:
import sys, inspect
repo_str = str(REPO)
sys.path = [p for p in sys.path if p != repo_str]
sys.path.insert(0, repo_str)
for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
from src.planning import ResearchPlanner
print('Planner signature:', inspect.signature(ResearchPlanner.execute))
assert 'lessons' in inspect.signature(ResearchPlanner.execute).parameters

Planner signature: (self, ticker: 'str', research_question: 'str', lessons: 'list | None' = None) -> 'dict[str, Any]'


## 4. Planner and Qwen execution

The planner routes the AAPL research question through market, earnings, and news specialists, then assembles evidence for Qwen draft generation. `TEST_LESSONS` are explicit test inputs rather than lessons retrieved from persistent memory. The model prompt and decoding behavior are defined in the existing integration source, not reimplemented in this notebook. Prior integration testing used `Qwen/Qwen2.5-1.5B-Instruct` with `max_new_tokens=512` and `do_sample=False`; these settings should be checked against the integration source before comparison.

In [4]:
TICKER = 'AAPL'
QUESTION = "Summarize Apple's market performance, financial metrics, and recent news."
TEST_LESSONS = [
    'Do not invent currency, financial units, or reporting periods.',
    'Cite source URLs and distinguish headlines from verified facts.',
]

# Integration entry point: returns plan, evidence, draft, and specialist trace.
result = ResearchPlanner().execute(
    ticker=TICKER,
    research_question=QUESTION,
    lessons=TEST_LESSONS,
)

required = {'evidence', 'draft_analysis', 'plan', 'retrieved_lessons', 'tool_trace'}
assert required.issubset(result), f'Missing fields: {required - set(result)}'
print('Live run complete. Result keys:', list(result))

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Live run complete. Result keys: ['ticker', 'research_question', 'specialists', 'evidence', 'draft_analysis', 'plan', 'retrieved_lessons', 'tool_trace']


## 5. Execution outputs and diagnostics

The returned plan, tool trace, and generated draft are displayed separately to preserve the distinction between routing, specialist execution, and language-model generation. A planned evaluation handoff does not constitute execution of the separate evaluator.

In [5]:
import json
from IPython.display import display, Markdown

print('PLANNER ROUTING')
print(json.dumps(result['plan'], indent=2, default=str))
print('\nTOOL TRACE')
for step in result['tool_trace']:
    print(step.get('specialist', step.get('tool', 'unknown')), '| errors:', step.get('errors', []))
print('\nQWEN DRAFT')
draft = result['draft_analysis']
if isinstance(draft, dict):
    display(Markdown(str(draft.get('text', draft))))
else:
    display(Markdown(str(draft)))

PLANNER ROUTING
{
  "ticker": "AAPL",
  "research_question": "Summarize Apple's market performance, financial metrics, and recent news.",
  "specialists": [
    "market",
    "earnings",
    "news"
  ],
  "steps": [
    "Review retrieved lessons before collecting and drafting evidence: ['Do not invent currency, financial units, or reporting periods.', 'Cite source URLs and distinguish headlines from verified facts.']",
    "Retrieve evidence from the market specialist.",
    "Retrieve evidence from the earnings specialist.",
    "Retrieve evidence from the news specialist.",
    "Combine the retrieved evidence into a draft analysis.",
    "Send the draft analysis and supporting evidence for evaluation."
  ],
  "retrieved_lessons": [
    "Do not invent currency, financial units, or reporting periods.",
    "Cite source URLs and distinguish headlines from verified facts."
  ]
}

TOOL TRACE
market | errors: []
earnings | errors: []
news | errors: []

QWEN DRAFT


Market Performance:
- Recent close: $343.46 per share
- 1-month return percentage: +1.92%

Financial Metrics:
- Total Revenue: $466,822,987,776
- Gross Profits: $227,123,003,392
- EBITDA: $167,959,003,136
- Net Income to Common Shareholders: $128,929,996,800

News:
- Positive sentiment towards Apple's new home collection giving investors another reason to remain bullish.
- Neutral sentiments regarding replacing certain stocks with others in an income portfolio.
- Negative sentiments about Apple fighting against Britain over iPhone tap-to-pay fees.
- Neutral sentiments on Elon Musk's Roadster, Nvidia's space AI push, and Microsoft's security warning.
- Neutral sentiments on Starlink mobile potentially affecting Apple.

Limitations:
- Data sources include Yahoo Finance historical prices and financials, Motley Fool, 24/7 Wall St., Investor's Business Daily, Barrons.com, Stocktwits, Simply Wall St., Investing.com, and Simply Wall St. articles.
- Sentiments range from neutral to positive across various topics related to Apple's products, strategies, and industry developments.

## 6. Evidence inspection

Market and earnings evidence is displayed alongside the generated draft to support numeric grounding checks. News sentiment labels are aggregated from the retrieved news items. Provider data can vary across runs; this section reports live values without assuming they match historical observations.

In [6]:
evidence = result['evidence']
print('Evidence sections:', list(evidence))
for section in ('market', 'earnings'):
    print(f'\n{section.upper()} EVIDENCE')
    print(json.dumps(evidence.get(section, {}), indent=2, default=str)[:4500])
news = evidence.get('news', {})
items = news.get('items', []) if isinstance(news, dict) else []
from collections import Counter
print('\nNews items:', len(items))
print('FinBERT labels:', dict(Counter(item.get('sentiment', 'unknown') for item in items)))
for item in items[:3]:
    print('-', item.get('title'), '|', item.get('sentiment'), '|', item.get('url'))

Evidence sections: ['market', 'earnings', 'news']

MARKET EVIDENCE
{
  "tool": "market_snapshot",
  "ticker": "AAPL",
  "data": {
    "period": "1mo",
    "as_of": "2026-10-16 00:00:00-04:00",
    "latest_close": 343.46,
    "period_return_pct": 1.92,
    "observations": 22
  },
  "sources": [
    {
      "name": "Yahoo Finance historical prices",
      "url": "https://finance.yahoo.com/quote/AAPL/history",
      "accessed_at": "2026-10-17T14:55:56.525717+00:00"
    }
  ],
  "errors": []
}

EARNINGS EVIDENCE
{
  "tool": "financial_snapshot",
  "ticker": "AAPL",
  "data": {
    "totalRevenue": 466822987776,
    "grossProfits": 227123003392,
    "ebitda": 167959003136,
    "netIncomeToCommon": 128929996800,
    "trailingPE": 39.38761,
    "profitMargins": 0.27618998
  },
  "sources": [
    {
      "name": "Yahoo Finance financials",
      "url": "https://finance.yahoo.com/quote/AAPL/financials",
      "accessed_at": "2026-10-17T14:55:57.807477+00:00"
    }
  ],
  "errors": []
}

News ite

## 7. Structured output for downstream integration

The full live result is serialized to JSON, preserving planner routing, evidence, generated analysis, and diagnostic fields. This artifact can be used for comparison against integration fixes or passed to a downstream evaluation adapter. No pre-uploaded JSON is required.

In [7]:
import json
output_path = Path('/content/person2_aapl_live_handoff.json')
output_path.write_text(json.dumps(result, indent=2, default=str), encoding='utf-8')
print('Saved:', output_path, '| bytes:', output_path.stat().st_size)
# Optional manual download in Colab:
# from google.colab import files
# files.download(str(output_path))

Saved: /content/person2_aapl_live_handoff.json | bytes: 20638


## 8. Historical integration validation and limitations

A prior complete AAPL test at integration commit `de11239` produced a market close of $343.20 and one-month return of +3.58%; both matched the generated draft. Four displayed financial values also matched. The ten retrieved news items had three positive, six neutral, and one negative sentiment labels. No specialist tool errors were reported.

**Observed limitations:** The Qwen draft omitted individual source URLs, financial output had minor Markdown formatting issues, and at least one retrieved headline was not directly about AAPL. News claims were not fully verified against all headlines. These are historical findings, not assertions about a new execution.

**Downstream boundary:** Ashley's `run_evaluation_revision(context, call_model, draft=None, lessons=None, max_revisions=2, run_id=None)` and persistent-memory functions are integrated separately. The present notebook provides the planner result and Qwen draft but does not claim evaluation, revision, or memory-test results.